# Phương pháp Thu thập Dữ liệu Bài báo Khoa học (Crawling Methodology)

> **Môn học:** Nhập môn Khoa học Dữ liệu (Introduction to Data Science) — HCMUS  
> **Milestone 1:** Data Crawling & Performance Benchmarking  
> **Tác giả:** Nhóm 04 (Anh Tuấn - MSSV 24120153)  
> **Ngày cập nhật:** Tháng 10/2026

---

## 1. Mục tiêu và Phạm vi Thu thập (Scope & Objectives)

Mục tiêu của Milestone 1 là xây dựng một quy trình thu thập dữ liệu khoa học tự động, mạnh mẽ và có thể tái lập, nhằm chuẩn bị tập dữ liệu bài báo phục vụ cho bài toán khai phá dữ liệu văn bản và trích xuất mạng lưới trích dẫn (Milestone 2 & 3).

### Yêu cầu cốt lõi theo đề bài:
1. **Metadata bài báo:** Thu thập đầy đủ các trường thông tin theo bảng `papers` (arXiv ID, Title, Authors, Submission Date, Revised Dates, Venue).
2. **Toàn văn mã nguồn TeX (Full-text Source):** Tải về toàn bộ các version có sẵn của từng bài báo từ arXiv mirror (`export.arxiv.org/src/`), giải nén và giữ lại duy nhất các file `.tex` và `.bib` (loại bỏ hoàn toàn các file ảnh, PDF, và file nhị phân trung gian).
3. **Mạng lưới trích dẫn (Citation Network):** Khai thác danh mục tài liệu tham khảo (references) từ Semantic Scholar Graph API cho từng bài báo tương ứng.
4. **Đo lường hiệu năng (Benchmarking):** Ghi nhận có hệ thống 4 chỉ số tài nguyên: End-to-end wall-clock time, Peak RAM, Average RAM và Disk usage delta.


---

## 2. Khảo sát và So sánh các Nguồn Dữ liệu (Data Sources Evaluation)

Để thu thập kho dữ liệu bài báo khoa học quy mô lớn, chúng tôi khảo sát 4 nguồn tài nguyên học thuật chính:
1. **arXiv API (Atom 1.0 Feed)**
2. **arXiv OAI-PMH (Open Archives Initiative Protocol for Metadata Harvesting)**
3. **Kaggle arXiv Dataset (Bulk Snapshot)**
4. **Semantic Scholar Graph API**

### Bảng so sánh đặc tính kỹ thuật giữa 4 nguồn:

| Đặc tính kỹ thuật | arXiv API (Atom) | arXiv OAI-PMH | Kaggle arXiv Dataset | Semantic Scholar Graph API |
|---|---|---|---|---|
| **Giao thức / Định dạng** | REST over HTTP / XML Atom 1.0 | REST over HTTP / XML Dublin Core & arXiv Raw | JSON Lines file dump | REST over HTTP / JSON |
| **Endpoint chính thức** | `https://export.arxiv.org/api/query` | `https://oaipmh.arxiv.org/oai` *(mới 2025)* | Tải trực tiếp từ Kaggle Datasets | `https://api.semanticscholar.org/graph/v1/` |
| **Giới hạn tốc độ (Rate Limit)** | $\ge 3.0$ giây / request, 1 kết nối đồng thời | Pacing tương tự, dùng Resumption Token | Không (offline file processing) | 1 request / giây (không API key), 429 khi vượt |
| **Độ trễ cập nhật (Freshness)** | Thời gian thực (ngay khi công bố) | Định kỳ theo batch cập nhật | Snapshot định kỳ (trễ vài tuần/tháng) | Gần thời gian thực (sau vài ngày lập chỉ mục) |
| **Khả năng lấy ngày version** | **Chỉ trả $v_1$ (`published`) & bản mới (`updated`)** | **Đầy đủ ngày của mọi version trung gian** | Chỉ chứa bản mới nhất hoặc snapshot | Không phụ trách version arXiv |
| **Hỗ trợ tải TeX Source** | Qua mirror `/src/{versioned_id}` | Chỉ trả metadata | Không có TeX source | Không có TeX source |
| **Đồ thị trích dẫn (References)** | Không hỗ trợ trực tiếp | Không hỗ trợ | Không hỗ trợ | **Hỗ trợ mạnh mẽ qua `/paper/arXiv:{id}`** |
| **Tài liệu chính thức** | [arXiv API User Manual](https://arxiv.org/help/api/index) | [arXiv OAI-PMH Documentation](https://arxiv.org/help/oa/index) | [Kaggle Dataset Card](https://www.kaggle.com/datasets/Cornell-University/arxiv) | [S2 API Reference](https://api.semanticscholar.org/api-docs/graph) |


---

## 3. Phân tích Chi tiết Từng Nguồn Dữ liệu

### 3.1. arXiv API (Atom 1.0 Feed)
- **Cơ chế hoạt động:** Truy vấn thông qua HTTP GET với tham số `id_list` hoặc `search_query`. Phản hồi trả về tài liệu XML chuẩn Atom 1.0.
- **Ưu điểm:** Nhẹ, phản hồi nhanh, hỗ trợ tìm kiếm theo ID cụ thể được giao trong danh sách `ids/<mssv>.txt`.
- **Hạn chế kỹ thuật quan trọng:**
  > ⚠️ **Hạn chế về `revised_dates`:** arXiv Atom feed chỉ cung cấp 2 mốc thời gian: `published` (ngày nộp bản đầu tiên $v_1$) và `updated` (ngày cập nhật bản mới nhất). Đối với các bài báo có từ 3 version trở lên ($n \ge 3$), ngày của các version trung gian ($v_2, v_3, \dots, v_{n-1}$) **hoàn toàn không có trong phản hồi Atom**.
  > Nhóm ghi nhận trường hợp này với cờ `"revised_dates_complete": false` để đảm bảo tính trung thực của dữ liệu.

### 3.2. OAI-PMH (Open Archives Initiative Protocol for Metadata Harvesting)
- **Cơ chế hoạt động:** Sử dụng các động từ (verbs) chuẩn của giao thức OAI-PMH: `ListRecords`, `GetRecord`, `Identify`, `ListSets`.
- **Cập nhật quan trọng (2025):** Endpoint cũ `export.arxiv.org/oai2` đã chính thức ngừng hoạt động. arXiv đã chuyển toàn bộ dịch vụ OAI-PMH sang máy chủ mới tại `https://oaipmh.arxiv.org/oai`.
- **Giá trị khai thác:** Metadata định dạng `arXivRaw` trong OAI-PMH chứa lịch sử chi tiết (full version history) bao gồm kích thước và ngày nộp chính xác của từng version $v_1, v_2, \dots, v_n$. Đây là giải pháp nâng cao (Module A3) để hoàn thiện các trường `revised_dates` còn thiếu.

### 3.3. Kaggle arXiv Dataset
- **Cơ chế hoạt động:** Bản dump nén dạng JSON lines (khoảng 4GB nén, 14GB giải nén) chứa metadata của hơn 2 triệu bài báo arXiv.
- **Giá trị khai thác:** Cho phép cross-validation (đối chiếu chéo) dữ liệu metadata đã crawl mà không tốn chi phí băng thông hay vi phạm rate limit của arXiv server.

### 3.4. Semantic Scholar Graph API
- **Cơ chế hoạt động:** Truy vấn REST API qua mã định danh `arXiv:<id>`.
  - Endpoint: `https://api.semanticscholar.org/graph/v1/paper/arXiv:{arxiv_id}?fields=references.title,references.authors,references.publicationDate,references.venue,references.year,references.externalIds`
- **Giá trị khai thác:** Cung cấp toàn bộ danh mục tài liệu được trích dẫn (references), bao gồm cả bài báo có arXiv ID và bài báo chỉ có DOI / S2 ID.


---

## 4. Chuẩn hóa Schema và Ánh xạ Trường Dữ liệu (Schema Normalization)

Để đảm bảo tính nhất quán xuyên suốt đồ án, nhóm đã thống nhất hợp đồng dữ liệu tại `docs/schema.md`.

### 4.1. Quy ước Khóa định danh Publication:
- `arxiv_id`: Mã bài báo gốc (không có version), ví dụ `2310.12345` hoặc `hep-th/9901001`. Giữ nguyên trong mọi file JSON.
- `version`: Số nguyên chỉ phiên bản ($1, 2, 3$).
- `versioned_id`: ID kèm phiên bản (ví dụ `2310.12345v2`), chỉ dùng khi tải mã nguồn qua mirror `/src/`.
- `pub_key`: Tên thư mục an toàn trên đĩa (chuẩn hóa `.` thành `-` và `/` thành `_`), ví dụ `2310-12345` hoặc `hep-th_9901001`.

### 4.2. Bảng ánh xạ trường từ các nguồn vào bảng `papers`:

| Trường chuẩn (`papers`) | Kiểu dữ liệu | Nguồn arXiv API | Nguồn OAI-PMH | Nguồn Semantic Scholar | Quy tắc chuẩn hóa |
|---|---|---|---|---|---|
| `arxiv_id` | `VARCHAR(32)` | `entry_id` (tách phần đuôi) | `<id>` trong header | `externalIds.ArXiv` | Cắt bỏ URL prefix và suffix version |
| `title` | `TEXT` | `result.title` | `<title>` | `paper.title` | Ghép các dòng bị xuống dòng, loại bỏ khoảng trắng thừa |
| `authors` | `TEXT[]` | `[a.name for a in result.authors]` | `<authors>/<author>` | `[a.name for a in paper.authors]` | Danh sách họ tên đầy đủ, rỗng nếu không có |
| `submission_date` | `TIMESTAMPTZ` | `result.published` | `<version version="v1">/<date>` | Không dùng | Chuẩn hóa sang định dạng ISO 8601 UTC |
| `revised_dates` | `TIMESTAMPTZ[]` | `[result.updated]` (chỉ bản cuối) | Danh sách ngày của các bản $v_2 \dots v_n$ | Không dùng | Danh sách chuỗi ISO 8601, `[]` nếu bài chỉ có 1 version |
| `venue` | `VARCHAR(255)` | `result.journal_ref` | `<journal-ref>` | `paper.venue` | Chuỗi tên tạp chí/hội nghị, `null` nếu là bản thảo chưa xuất bản |


---

## 5. Ràng buộc Pacing, Giới hạn Tốc độ và Giải thuật Kháng lỗi (Rate Limits & Fault Tolerance)

### 5.1. Ràng buộc cứng từ phía máy chủ:
- **arXiv Policy:** Bắt buộc chờ tối thiểu **3.0 giây** giữa 2 yêu cầu liên tiếp và chỉ mở **1 kết nối đồng thời** (`REQUEST_DELAY = 3.0s`). Sử dụng header `User-Agent` mang định danh rõ ràng theo quy định:
  $$	ext{User-Agent: } 	ext{<GroupName>}/1.0 	ext{ (academic course project; mailto:<email>)}$$
- **Semantic Scholar Policy:** Giới hạn tối đa **1 request/giây** đối với tài khoản miễn phí (`S2_REQUEST_DELAY = 1.1s`).

### 5.2. Giải thuật Exponential Backoff with Jitter:
Khi gặp mã lỗi HTTP `429 (Too Many Requests)` hoặc các lỗi máy chủ tạm thời `5xx`, hệ thống áp dụng giải thuật chờ lũy thừa kết hợp nhiễu ngẫu nhiên (Full Jitter) để tránh xung đột đồng thời:

$$t_{	ext{wait}} = 	ext{random}(0, \min(t_{\max}, t_{	ext{base}} 	imes 2^{	ext{retry}}))$$

Dưới đây là mã nguồn minh họa giải thuật xử lý retry trong pipeline:


In [ ]:
import time
import random
import requests

def fetch_with_backoff(url: str, headers: dict, max_retries: int = 4, base_delay: float = 3.0):
    """Truy vấn HTTP với cơ chế Exponential Backoff with Jitter."""
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, headers=headers, timeout=30)
            if resp.status_code == 200:
                return resp
            elif resp.status_code in (429, 500, 502, 503, 504):
                # Tính thời gian chờ lũy thừa kèm ngẫu nhiên hóa (Full Jitter)
                cap = min(60.0, base_delay * (2 ** attempt))
                sleep_time = random.uniform(base_delay, cap)
                print(f"[Retry {attempt+1}/{max_retries}] HTTP {resp.status_code}. Backing off for {sleep_time:.2f}s...")
                time.sleep(sleep_time)
            else:
                resp.raise_for_status()
        except requests.RequestException as exc:
            if attempt == max_retries - 1:
                raise
            sleep_time = random.uniform(base_delay, base_delay * (2 ** attempt))
            print(f"[Retry {attempt+1}/{max_retries}] Lỗi kết nối: {exc}. Chờ {sleep_time:.2f}s...")
            time.sleep(sleep_time)
    raise RuntimeError(f"Vượt quá số lần retry cho URL: {url}")


---

## 6. Xử lý Toàn văn Source TeX (`/src/`) và Tối ưu Hóa Dung lượng

Đề bài yêu cầu tải toàn văn mã nguồn của **tất cả các version có sẵn** và giải nén có chọn lọc:

### 6.1. Quy trình xử lý source archive:
1. Gửi request tải archive từ: `https://export.arxiv.org/src/{versioned_id}`.
2. Kiểm tra `Content-Type` và magic bytes:
   - Nếu là `.tar.gz` hoặc `.tar`: Mở bằng thư viện `tarfile`.
   - Nếu là file `.gz` đơn: Giải nén bằng `gzip` thành 1 file `.tex`.
   - Nếu phản hồi trả về file PDF thuần túy: Ghi nhận version không có TeX source vào file `_EMPTY.json` với lý do `pdf_only`.
3. **Lọc dữ liệu nghiêm ngặt:** Chỉ trích xuất các file có đuôi `.tex` và `.bib`. Toàn bộ file ảnh (`.png`, `.jpg`, `.pdf`, `.eps`), style (`.sty`, `.cls`), và file trung gian (`.bbl`, `.aux`, `.log`) đều bị loại bỏ nhằm giảm thiểu 70-90% dung lượng lưu trữ trên đĩa.

```
Archive gốc (~2MB - 10MB) ──► Giải nén chọn lọc ──► .tex + .bib (~100KB - 500KB)
                                               └──► Bỏ ảnh & PDF thừa
```


---

## 7. Khả năng Tái lập và Hạ tầng Ghi nhận Log (Reproducibility & Logging)

Mọi hoạt động trong pipeline đều tuân thủ nguyên tắc **Idempotence (Tính bất biến khi chạy lại)** và được ghi nhận chi tiết vào thư mục `logs/`:
- `logs/discovery.csv`: Nhật ký phân giải metadata bài báo.
- `logs/download.csv`: Nhật ký tải mã nguồn từng version.
- `logs/extract.csv`: Thống kê dung lượng archive gốc (`bytes_before`) và dung lượng giữ lại (`bytes_after`).
- `logs/references.csv`: Thống kê số lượng reference thu thập từ Semantic Scholar.
- `logs/benchmark_runs.csv`: Nhật ký đo lường tài nguyên toàn diện (Wall-clock time, Peak RAM, Average RAM, Disk delta) phục vụ đối sánh và tối ưu hóa hệ thống.

---

## 8. Tài liệu Tham khảo Chính thức

1. arXiv API User Manual: [https://arxiv.org/help/api/index](https://arxiv.org/help/api/index)
2. arXiv OAI-PMH Documentation: [https://arxiv.org/help/oa/index](https://arxiv.org/help/oa/index)
3. Semantic Scholar API Documentation: [https://api.semanticscholar.org/api-docs/graph](https://api.semanticscholar.org/api-docs/graph)
4. Kaggle arXiv Dataset: [https://www.kaggle.com/datasets/Cornell-University/arxiv](https://www.kaggle.com/datasets/Cornell-University/arxiv)
